# Cadence - Encoder Pipeline

## 1. Install dependencies and check notebook runtime

In [1]:
%pip install --upgrade torch torchvision torchaudio torchao transformers litert-torch ai-edge-litert soundfile -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 74.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 97.6 MB/s eta 0:00:00


In [2]:
import csv
import importlib.util
import shutil
import subprocess
from pathlib import Path

REQUIRED_PACKAGES = [
    "torch",
    "transformers",
    "litert_torch",
    "ai_edge_litert",
    "soundfile",
    "numpy",
]
missing = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]
if missing:
    raise RuntimeError(
        f"Missing required packages: {missing}. Run the install cell above, then restart the kernel if needed."
    )

MANIFEST_PATH = Path("/content/cadence_dataset_samples/metadata/dataset_manifest.csv")
WORK_DIR = Path("/content/cadence_encoder_work")
LOCAL_WAV_DIR = WORK_DIR / "manifest_wavs"
OUTPUT_DIR = Path("/content/cadence_outputs")
MAX_ENCODER_SAMPLES = None

OUTPUT_MODEL_PATH = OUTPUT_DIR / "encoder_wav2vec2.tflite"
QUANT_MODEL_PATH = OUTPUT_DIR / "encoder_wav2vec2_int8.tflite"

WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_WAV_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Manifest: {MANIFEST_PATH}")
print(f"Working files: {WORK_DIR}")
print(f"Final model: {OUTPUT_MODEL_PATH}")

if shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(
        result.stdout
        if result.returncode == 0
        else "nvidia-smi is available but did not report a GPU."
    )
else:
    print("nvidia-smi not found. PyTorch CUDA availability is checked later.")

Manifest: /content/cadence_dataset_samples/metadata/dataset_manifest.csv
Working files: /content/cadence_encoder_work
Final model: /content/cadence_outputs/encoder_wav2vec2.tflite
Tue Apr 21 23:55:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             31W /   70W |     113M

## 2. Mount Google Drive and load sampled WAV manifest

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Dataset manifest not found: {MANIFEST_PATH}. "
        "Run preprocessing_pipeline_random_samples.ipynb first, or point MANIFEST_PATH at the Drive manifest copy."
    )

In [5]:
def load_completed_manifest_rows(manifest_path: Path) -> list[dict[str, str]]:
    with manifest_path.open("r", newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))

    completed_rows = []
    for row in rows:
        status = row.get("status", "")
        wav_drive_path = row.get("wav_drive_path", "")
        if not status.startswith("completed"):
            continue
        if not wav_drive_path:
            continue
        if not Path(wav_drive_path).exists():
            raise FileNotFoundError(
                f"Drive WAV path missing for {row.get('sample_id')}: {wav_drive_path}"
            )
        completed_rows.append(row)

    if MAX_ENCODER_SAMPLES is not None:
        completed_rows = completed_rows[:MAX_ENCODER_SAMPLES]

    if not completed_rows:
        raise RuntimeError(
            "No completed manifest rows with wav_drive_path were found. "
            "The encoder pipeline intentionally uses Drive WAVs as the source of truth."
        )

    return completed_rows


manifest_rows = load_completed_manifest_rows(MANIFEST_PATH)
print(f"Loaded {len(manifest_rows)} manifest WAV rows for encoder verification.")
for row in manifest_rows:
    print(f"- {row['sample_id']}: {row['wav_drive_path']}")

Loaded 10 manifest WAV rows for encoder verification.
- sample_001_26_20-09-2025_Practical_P.4_-_Part_1_-_Seasons_Calculator_Functions_Procedures: /content/drive/MyDrive/Cadence/Samples/wav/sample_001_26_20-09-2025_Practical_P.4_-_Part_1_-_Seasons_Calculator_Functions_Procedures.wav
- sample_002_14_09-08-2025_Practical_P.3_-_Part_2_-_WHILE_FOR_Loops: /content/drive/MyDrive/Cadence/Samples/wav/sample_002_14_09-08-2025_Practical_P.3_-_Part_2_-_WHILE_FOR_Loops.wav
- sample_003_46_29-11-2025_Practical_P.6_-_Part_1_-_File_Handling_in_Python: /content/drive/MyDrive/Cadence/Samples/wav/sample_003_46_29-11-2025_Practical_P.6_-_Part_1_-_File_Handling_in_Python.wav
- sample_004_41_13-11-2025_Theory_15.2_-_Part_2_-_Boolean_Simplification_Exercises: /content/drive/MyDrive/Cadence/Samples/wav/sample_004_41_13-11-2025_Theory_15.2_-_Part_2_-_Boolean_Simplification_Exercises.wav
- sample_005_39_06-11-2025_Theory_15.2_-_Part_1_-_Logic_Gates_Boolean_Algebra_Laws: /content/drive/MyDrive/Cadence/Samples/w

## 3. Stage manifest WAV samples locally

In [6]:
staged_wavs = []

for row in manifest_rows:
    source_path = Path(row["wav_drive_path"])
    local_path = LOCAL_WAV_DIR / source_path.name
    shutil.copy2(source_path, local_path)
    staged_wavs.append(local_path)
    size_mb = local_path.stat().st_size / (1024 * 1024)
    print(f"OK {row['sample_id']} -> {local_path.name} ({size_mb:.1f} MB)")

print(f"Staged {len(staged_wavs)} WAV files from Drive into {LOCAL_WAV_DIR}.")

OK sample_001_26_20-09-2025_Practical_P.4_-_Part_1_-_Seasons_Calculator_Functions_Procedures -> sample_001_26_20-09-2025_Practical_P.4_-_Part_1_-_Seasons_Calculator_Functions_Procedures.wav (18.6 MB)
OK sample_002_14_09-08-2025_Practical_P.3_-_Part_2_-_WHILE_FOR_Loops -> sample_002_14_09-08-2025_Practical_P.3_-_Part_2_-_WHILE_FOR_Loops.wav (20.4 MB)
OK sample_003_46_29-11-2025_Practical_P.6_-_Part_1_-_File_Handling_in_Python -> sample_003_46_29-11-2025_Practical_P.6_-_Part_1_-_File_Handling_in_Python.wav (23.4 MB)
OK sample_004_41_13-11-2025_Theory_15.2_-_Part_2_-_Boolean_Simplification_Exercises -> sample_004_41_13-11-2025_Theory_15.2_-_Part_2_-_Boolean_Simplification_Exercises.wav (24.7 MB)
OK sample_005_39_06-11-2025_Theory_15.2_-_Part_1_-_Logic_Gates_Boolean_Algebra_Laws -> sample_005_39_06-11-2025_Theory_15.2_-_Part_1_-_Logic_Gates_Boolean_Algebra_Laws.wav (22.4 MB)
OK sample_006_29_02-10-2025_Theory_14.1_-_Part_2_-_TCP-IP -> sample_006_29_02-10-2025_Theory_14.1_-_Part_2_-_TCP-IP.

## 4. Load and freeze Wav2Vec2 encoder

In [7]:
import torch
from transformers import Wav2Vec2Model

MODEL_NAME = "facebook/wav2vec2-base-960h"
print(f"Loading {MODEL_NAME}.")

encoder = Wav2Vec2Model.from_pretrained(MODEL_NAME)
encoder.eval()

for param in encoder.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in encoder.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Model size (float32): ~{total_params * 4 / 1024 / 1024:.1f} MB")
print(f"Model size (int8 estimate): ~{total_params / 1024 / 1024:.1f} MB")
print("Encoder loaded and frozen.")

Loading facebook/wav2vec2-base-960h.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 94,371,712
Model size (float32): ~360.0 MB
Model size (int8 estimate): ~90.0 MB
Encoder loaded and frozen.


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

encoder = encoder.to(device).eval()

Using device: cuda
CUDA device: Tesla T4


## 5. Inspect embeddings on sample classroom audio

In [9]:
import numpy as np
import soundfile as sf

CHUNK_SAMPLES = 160_000


def load_wav_chunk(wav_path, chunk_index=0):
    audio, sr = sf.read(str(wav_path), dtype="float32", always_2d=False)
    assert sr == 16000, f"Expected 16kHz, got {sr}Hz in {wav_path}"

    if audio.ndim > 1:
        audio = np.mean(audio, axis=1).astype(np.float32)

    start = chunk_index * CHUNK_SAMPLES
    end = start + CHUNK_SAMPLES
    chunk = audio[start:end]

    if len(chunk) < CHUNK_SAMPLES:
        chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)))

    return chunk.astype(np.float32)


def analyse_embeddings(embeddings_np):
    embedding_std = float(np.mean(np.std(embeddings_np, axis=0)))

    norms = np.linalg.norm(embeddings_np, axis=1, keepdims=True)
    normalised = embeddings_np / (norms + 1e-8)
    frame_sims = np.sum(normalised[:-1] * normalised[1:], axis=1)

    return {
        "shape": embeddings_np.shape,
        "embedding_std": round(embedding_std, 4),
        "mean_adjacent_cosine": round(float(np.mean(frame_sims)), 4),
        "global_mean": round(float(np.mean(embeddings_np)), 4),
        "global_std": round(float(np.std(embeddings_np)), 4),
        "has_nan": bool(np.any(np.isnan(embeddings_np))),
        "has_inf": bool(np.any(np.isinf(embeddings_np))),
    }

In [10]:
print("Running encoder on manifest classroom audio chunks.")
all_stats = []

for wav_path in staged_wavs:
    chunk = load_wav_chunk(wav_path, chunk_index=0)
    input_tensor = torch.from_numpy(chunk).unsqueeze(0).to(device)

    with torch.no_grad():
        output = encoder(input_values=input_tensor)
        embeddings_np = output.last_hidden_state.squeeze(0).cpu().numpy()

    stats = analyse_embeddings(embeddings_np)
    all_stats.append(stats)

    print(
        f"Filename: {wav_path.name}, "
        f"Shape: {stats['shape']}, "
        f"EmbeddingStd: {stats['embedding_std']}, "
        f"CosSim: {stats['mean_adjacent_cosine']}, "
        f"NaN: {'YES' if stats['has_nan'] else 'no'}, "
        f"Inf: {'YES' if stats['has_inf'] else 'no'}"
    )

if any(s["has_nan"] or s["has_inf"] for s in all_stats):
    raise RuntimeError(
        "NaN or Inf detected in encoder embeddings. Check sampled WAV files."
    )

mean_embedding_std = float(np.mean([s["embedding_std"] for s in all_stats]))
print(f"Mean embedding std across {len(all_stats)} files: {mean_embedding_std:.4f}.")
print("All embeddings clean. Frozen encoder verified on sampled manifest audio.")

Running encoder on manifest classroom audio chunks.
Filename: sample_001_26_20-09-2025_Practical_P.4_-_Part_1_-_Seasons_Calculator_Functions_Procedures.wav, Shape: (499, 768), EmbeddingStd: 0.1428, CosSim: 0.8028, NaN: no, Inf: no
Filename: sample_002_14_09-08-2025_Practical_P.3_-_Part_2_-_WHILE_FOR_Loops.wav, Shape: (499, 768), EmbeddingStd: 0.1554, CosSim: 0.6581, NaN: no, Inf: no
Filename: sample_003_46_29-11-2025_Practical_P.6_-_Part_1_-_File_Handling_in_Python.wav, Shape: (499, 768), EmbeddingStd: 0.1657, CosSim: 0.8023, NaN: no, Inf: no
Filename: sample_004_41_13-11-2025_Theory_15.2_-_Part_2_-_Boolean_Simplification_Exercises.wav, Shape: (499, 768), EmbeddingStd: 0.1532, CosSim: 0.7403, NaN: no, Inf: no
Filename: sample_005_39_06-11-2025_Theory_15.2_-_Part_1_-_Logic_Gates_Boolean_Algebra_Laws.wav, Shape: (499, 768), EmbeddingStd: 0.1281, CosSim: 0.8355, NaN: no, Inf: no
Filename: sample_006_29_02-10-2025_Theory_14.1_-_Part_2_-_TCP-IP.wav, Shape: (499, 768), EmbeddingStd: 0.1448, 

## 6. Wrap encoder for fixed-shape export

In [11]:
import torch.nn as nn
import torch.nn.functional as F

TARGET_FRAMES = 500
EMBEDDING_DIM = 768


class Wav2Vec2EncoderWrapper(nn.Module):
    def __init__(self, wav2vec2_model):
        super().__init__()
        self.encoder = wav2vec2_model

    def forward(self, x):
        # Input: float32[1, 160000]. Output: float32[1, 500, 768].
        output = self.encoder(input_values=x)
        embeddings = output.last_hidden_state

        time_frames = embeddings.shape[1]
        if time_frames < TARGET_FRAMES:
            embeddings = F.pad(embeddings, (0, 0, 0, TARGET_FRAMES - time_frames))
        elif time_frames > TARGET_FRAMES:
            embeddings = embeddings[:, :TARGET_FRAMES, :]

        return embeddings

In [12]:
wrapped_model = Wav2Vec2EncoderWrapper(encoder).eval()

dummy_input = torch.zeros(1, CHUNK_SAMPLES, dtype=torch.float32).to(device)
with torch.no_grad():
    test_output = wrapped_model(dummy_input)

print(f"Input shape: {tuple(dummy_input.shape)}")
print(f"Output shape: {tuple(test_output.shape)}")

assert tuple(test_output.shape) == (1, TARGET_FRAMES, EMBEDDING_DIM), (
    f"Shape mismatch: expected (1, {TARGET_FRAMES}, {EMBEDDING_DIM}), got {tuple(test_output.shape)}."
)
print("Wrapper shape contract verified.")

Input shape: (1, 160000)
Output shape: (1, 500, 768)
Wrapper shape contract verified.


## 7. INT8 quantisation + TFLite export

In [13]:
import torchao
import litert_torch

print("torchao:", torchao.__version__)
print("litert_torch import OK")

/usr/local/lib/python3.12/dist-packages/torch/distributed/distributed_c10d.py:351: UserWarning: Device capability of jax unspecified, assuming `cpu` and `cuda`. Please specify it via the `devices` argument of `register_backend`.
  warnings.warn(


torchao: 0.17.0
litert_torch import OK


In [14]:
wrapped_model_cpu = wrapped_model.cpu().eval()

sample_input = (torch.zeros(1, CHUNK_SAMPLES, dtype=torch.float32),)

print("Converting PyTorch to LiteRT/TFLite via litert_torch.")

edge_model = litert_torch.convert(wrapped_model_cpu, sample_input)
edge_model.export(str(OUTPUT_MODEL_PATH))

size_mb = OUTPUT_MODEL_PATH.stat().st_size / (1024 * 1024)
print(f"\nTFLite saved: {OUTPUT_MODEL_PATH} ({size_mb:.1f} MB)")

Converting PyTorch to LiteRT/TFLite via litert_torch.

TFLite saved: /content/cadence_outputs/encoder_wav2vec2.tflite (360.5 MB)


In [15]:
import tensorflow as tf

_ai_edge_converter_flags = {"optimizations": [tf.lite.Optimize.DEFAULT]}

print("Quantizing with built-in dynamic range quantisation.")

edge_model_int8 = litert_torch.convert(
    wrapped_model_cpu, sample_input, _ai_edge_converter_flags=_ai_edge_converter_flags
)
edge_model_int8.export(str(QUANT_MODEL_PATH))

size_int8_mb = QUANT_MODEL_PATH.stat().st_size / (1024 * 1024)
print(f"\nQuantised TFLite saved: {QUANT_MODEL_PATH} ({size_int8_mb:.1f} MB)")

Quantizing with built-in dynamic range quantisation.

Quantised TFLite saved: /content/cadence_outputs/encoder_wav2vec2_int8.tflite (91.9 MB)


## 8. Verify the INT8 TFLite model

In [16]:
import numpy as np
from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path=str(QUANT_MODEL_PATH))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

checks = {
    "input_name": input_details[0]["name"],
    "input_shape": tuple(input_details[0]["shape"].tolist()),
    "input_dtype": input_details[0]["dtype"],
    "output_name": output_details[0]["name"],
    "output_shape": tuple(output_details[0]["shape"].tolist()),
    "output_dtype": output_details[0]["dtype"],
}

print("TFLite Tensor Contract Verification")
print(f"Input name: {checks['input_name']}")
print(f"Input shape: {checks['input_shape']}")
print(f"Input dtype: {checks['input_dtype']}")
print(f"Output name: {checks['output_name']}")
print(f"Output shape: {checks['output_shape']}")
print(f"Output dtype: {checks['output_dtype']}")

zero_input = np.zeros((1, CHUNK_SAMPLES), dtype=np.float32)
interpreter.set_tensor(input_details[0]["index"], zero_input)
interpreter.invoke()
zero_output = interpreter.get_tensor(output_details[0]["index"])

real_chunk = load_wav_chunk(staged_wavs[0], chunk_index=0).reshape(1, -1)
interpreter.set_tensor(input_details[0]["index"], real_chunk)
interpreter.invoke()
real_output = interpreter.get_tensor(output_details[0]["index"])
real_embedding_std = float(np.mean(np.std(real_output.squeeze(0), axis=0)))

verification_results = [
    ("Input shape == (1, 160000)", checks["input_shape"] == (1, CHUNK_SAMPLES)),
    ("Input dtype == float32", checks["input_dtype"] == np.float32),
    (
        "Output shape == (1, 500, 768)",
        checks["output_shape"] == (1, TARGET_FRAMES, EMBEDDING_DIM),
    ),
    ("Output dtype == float32", checks["output_dtype"] == np.float32),
    ("No NaN in zero-input output", not bool(np.any(np.isnan(zero_output)))),
    ("No Inf in zero-input output", not bool(np.any(np.isinf(zero_output)))),
    ("No NaN in real-audio output", not bool(np.any(np.isnan(real_output)))),
    ("No Inf in real-audio output", not bool(np.any(np.isinf(real_output)))),
]
name_results = [
    ("Input name == input_values", checks["input_name"] == "input_values"),
    ("Output name == embeddings", checks["output_name"] == "embeddings"),
]

all_passed = True
names_passed = True

print("Verification Results")
for label, passed in verification_results:
    print(f"- [{'PASS' if passed else 'FAIL'}] {label}")
    all_passed = all_passed and passed

for label, passed in name_results:
    print(f"- [{'PASS' if passed else 'INFO'}] {label}")
    names_passed = names_passed and passed

print(f"Real-audio embedding std: {real_embedding_std:.4f}")

if not all_passed:
    raise RuntimeError("TFLite verification failed.")

if not names_passed:
    print("Tensor names differ from the original contract.")

print("TFLite verification passed.")

TFLite Tensor Contract Verification
Input name: serving_default_args_0:0
Input shape: (1, 160000)
Input dtype: <class 'numpy.float32'>
Output name: StatefulPartitionedCall:0
Output shape: (1, 500, 768)
Output dtype: <class 'numpy.float32'>
Verification Results
- [PASS] Input shape == (1, 160000)
- [PASS] Input dtype == float32
- [PASS] Output shape == (1, 500, 768)
- [PASS] Output dtype == float32
- [PASS] No NaN in zero-input output
- [PASS] No Inf in zero-input output
- [PASS] No NaN in real-audio output
- [PASS] No Inf in real-audio output
- [INFO] Input name == input_values
- [INFO] Output name == embeddings
Real-audio embedding std: 0.1275
Tensor names differ from the original contract.
TFLite verification passed.
